# 第6节：帧率、码率与时间戳

本 Notebook 包含三个实验，帮助你理解帧率、码率和时间戳的概念。

**实验内容：**
1. 提取并分析时间戳
2. 分析 GOP 结构
3. 码率分析

## 环境准备

确保已安装 ffmpeg、ffprobe。

In [ ]:
import subprocess
import json
import os
import time

# 检查 ffmpeg、ffprobe 是否可用
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '-version'], capture_output=True, text=True, timeout=10)
        version = result.stdout.split('\n')[0]
        print(f"✓ {cmd} 已安装: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {cmd} 未安装")
        return False
    except subprocess.TimeoutExpired:
        print(f"✗ {cmd} 执行超时")
        return False

check_command('ffmpeg')
check_command('ffprobe')

## 生成测试素材

生成带 B 帧的测试视频（每秒1个关键帧，B帧数=2）。

In [ ]:
def run_ffmpeg_cmd(cmd, description, timeout=30):
    """执行 ffmpeg 命令并检查结果"""
    print(f"  {description}...")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"  ✗ 失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  ✗ 超时 ({timeout}秒)")
        return False

# 生成带 B 帧的测试视频
print("生成测试素材中...")

cmd = [
    'ffmpeg',
    '-f', 'lavfi', '-i', 'testsrc=duration=5:size=1280x720:rate=30',
    '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
    '-c:v', 'libx264', '-bf', '2', '-g', '30',
    '-c:a', 'aac', '-shortest', '-y', 'test_bframes.mp4'
]

if run_ffmpeg_cmd(cmd, "生成测试视频"):
    size = os.path.getsize('test_bframes.mp4') / 1024
    print(f"\n✓ 测试视频生成成功: {size:.1f} KB")

## 实验1：提取并分析时间戳

**目标**：理解 PTS 和 DTS 的含义

In [ ]:
def get_frame_timestamps(file_path, timeout=30):
    """获取每一帧的时间戳信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'frame=pts,pts_time,dts,dts_time,pict_type',
        '-read_intervals', '%+5',  # 只读取前5秒
        '-of', 'json',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 分析带 B 帧的视频
print("=" * 60)
print("时间戳分析")
print("=" * 60)

info = get_frame_timestamps('test_bframes.mp4')
if info and 'frames' in info:
    frames = info['frames']
    
    print(f"\n总帧数 (前5秒): {len(frames)}")
    print(f"\n前10帧的时间戳:")
    print(f"{'帧类型':<8} {'PTS':<10} {'PTS时间':<12} {'DTS':<10} {'DTS时间':<12}")
    print("-" * 52)
    
    for i, frame in enumerate(frames[:10]):
        pict_type = frame.get('pict_type', 'N/A')
        pts = frame.get('pts', 'N/A')
        pts_time = frame.get('pts_time', 'N/A')
        dts = frame.get('dts', 'N/A')
        dts_time = frame.get('dts_time', 'N/A')
        
        # 格式化时间
        if pts_time != 'N/A':
            pts_time = f"{float(pts_time):.6f}"
        if dts_time != 'N/A':
            dts_time = f"{float(dts_time):.6f}"
        
        print(f"{pict_type:<8} {pts:<10} {pts_time:<12} {dts:<10} {dts_time:<12}")

## 实验2：分析 GOP 结构

**目标**：理解 GOP 的组成

In [ ]:
def analyze_gop(file_path, timeout=30):
    """分析 GOP 结构"""
    info = get_frame_timestamps(file_path, timeout)
    if not info or 'frames' not in info:
        return None
    
    frames = info['frames']
    gops = []
    current_gop = []
    
    for frame in frames:
        current_gop.append(frame)
        if frame.get('pict_type') == 'I' and len(current_gop) > 1:
            # 遇到新的 I 帧，保存当前 GOP
            gops.append(current_gop[:-1])
            current_gop = [frame]
    
    # 保存最后一个 GOP
    if current_gop:
        gops.append(current_gop)
    
    return gops

# 分析 GOP 结构
print("=" * 60)
print("GOP 结构分析")
print("=" * 60)

gops = analyze_gop('test_bframes.mp4')
if gops:
    print(f"\nGOP 数量: {len(gops)}")
    
    for i, gop in enumerate(gops[:3]):  # 只显示前3个 GOP
        print(f"\nGOP {i + 1}:")
        print(f"  帧数: {len(gop)}")
        print(f"  帧类型: ", end="")
        for frame in gop:
            print(frame.get('pict_type', '?'), end=" ")
        print()
        
        # 显示时间范围
        if gop:
            start_time = gop[0].get('pts_time', 'N/A')
            end_time = gop[-1].get('pts_time', 'N/A')
            if start_time != 'N/A' and end_time != 'N/A':
                print(f"  时间范围: {float(start_time):.3f}s - {float(end_time):.3f}s")

## 实验3：码率分析

**目标**：理解码率与画质的关系

In [ ]:
def get_bitrate_info(file_path, timeout=10):
    """获取码率信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        '-show_streams',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 分析码率信息
print("=" * 60)
print("码率分析")
print("=" * 60)

info = get_bitrate_info('test_bframes.mp4')
if info:
    # 格式信息
    fmt = info['format']
    print(f"\n容器格式: {fmt['format_name']}")
    print(f"时长: {float(fmt['duration']):.2f} 秒")
    print(f"总码率: {int(fmt['bit_rate']) / 1000:.0f} kbps")
    
    # 流信息
    for stream in info['streams']:
        if stream['codec_type'] == 'video':
            print(f"\n视频流:")
            print(f"  编码: {stream['codec_name']}")
            print(f"  分辨率: {stream['width']}x{stream['height']}")
            print(f"  帧率: {stream['r_frame_rate']}")
            print(f"  码率: {int(stream.get('bit_rate', 0)) / 1000:.0f} kbps")
        elif stream['codec_type'] == 'audio':
            print(f"\n音频流:")
            print(f"  编码: {stream['codec_name']}")
            print(f"  采样率: {stream['sample_rate']} Hz")
            print(f"  码率: {int(stream.get('bit_rate', 0)) / 1000:.0f} kbps")

## 总结

通过本实验，你应该掌握了：

1. **帧率、码率、时间戳的概念**
   - 帧率决定流畅度，码率决定画质
   - PTS 决定显示时间，DTS 决定解码时间

2. **B 帧的影响**
   - 有 B 帧时，PTS 和 DTS 可能不同
   - 解码顺序和显示顺序不同

3. **GOP 结构**
   - GOP 是两个关键帧之间的帧序列
   - GOP 越大，压缩率越高，但 seek 精度越低

4. **码率分析**
   - 码率越高画质越好，但带宽需求也越大
   - 直播用 CBR，点播用 VBR